# Extend the TurnBackHoax dataset

Simple, student-project version — no classes, no CLI flags, just run top to bottom.

**Workflow:** Kaggle > Add Data > search `dataset hoax turnbackhoax` (aginanjar) to attach the
existing 15,674-row archive (through Oct 2024) as input. This notebook picks up where it left
off and scrapes forward. Run it, then **Save Version > Save & Run All**, then
**New Dataset** from the output CSV to publish your extended version — same habit as always,
just pointed at a source that's actually allowed to be scraped (see cell 2).

No existing dataset to extend? Skip cell 3 and set `START_ID` manually — turnbackhoax.id
article ids are sequential, so any recent id (browse the site to find one) works as a start.

## 1. Setup

In [ ]:
import json, re, time, random
from html import unescape
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

BASE = "https://turnbackhoax.id"
OUT_CSV = Path("/kaggle/working/turnbackhoax_extended.csv")

# Say who you are and how to reach you — basic courtesy for a small student scraper.
HEADERS = {"User-Agent": "owi-student-project/0.1 (contact: sinambeladavid087@gmail.com)"}
DELAY = (1.5, 3.0)  # seconds between requests, randomized — don't hammer the site

## 2. Why this site, not Kompas/Detik

Checked `robots.txt` for the sites we considered:
- **kompas.com** explicitly bans automated scraping for dataset/ML use in its robots.txt
  comments and blocks `ClaudeBot`/`Scrapy`/etc. by name — skip it.
- **detik.com** allows crawling but has no fact-check section (`/cek-fakta` is a 404) —
  it's general news, not verdicts.
- **turnbackhoax.id** (Mafindo) has `Disallow:` (nothing blocked), and every article embeds
  `ClaimReview` JSON-LD — the same structured-data format Google Fact Check Tools uses — so
  claim/verdict/date are already machine-readable on the page.

## 3. Resume from the existing Kaggle dataset (optional)

If you added `aginanjar/dataset-hoax-turnbackhoax` via **Add Data**, this finds the highest
article id already covered so you only scrape what's new.

In [ ]:
existing_path = Path("/kaggle/input/dataset-hoax-turnbackhoax/hoax_dataset_with_text (2).csv")

if existing_path.exists():
    existing = pd.read_csv(existing_path)
    existing_ids = existing["link"].str.extract(r"/(\d{4})/(\d{2})/(\d{2})/")  # old URL scheme has no numeric id
    # The old dataset used date-based URLs (/2024/10/30/slug/), not the numeric /articles/{id}
    # scheme the live site uses now. We can't infer a numeric id from it, so just note the
    # latest date covered and start scraping from today's live articles going backward instead.
    latest_date = pd.to_datetime(existing["tanggal"], format="%B %d, %Y", errors="coerce").max()
    print(f"existing dataset covers up to ~{latest_date}; {len(existing)} rows")
    START_ID = None  # see cell 4: falls back to walking the live listing pages
else:
    print("no existing dataset attached — set START_ID manually below")
    START_ID = None

## 4. Fetch + parse

Two ways to discover articles:
- **`START_ID`/`END_ID` set** → walk that numeric range directly (fastest for a backfill;
  ids are sequential and the URL slug is decorative, `/articles/36622` resolves fine).
- **Neither set** → walk the most recent listing pages (`PAGES_TO_WALK`), good for "just get me
  what's new since the community dataset was made."

In [ ]:
START_ID = None      # e.g. 36400 to backfill a range
END_ID = None        # e.g. 36700
PAGES_TO_WALK = 15   # used only when START_ID is None

VERDICT_MAP = {
    "salah": "FALSE", "hoaks": "FALSE", "penipuan": "FALSE",
    "konten yang dimanipulasi": "FALSE", "konten palsu": "FALSE",
    "benar": "TRUE", "sebagian benar": "MISLEADING", "menyesatkan": "MISLEADING",
    "satire": "OPINION", "parodi": "OPINION",
}


def fetch(url, session, retries=3):
    for attempt in range(retries):
        try:
            resp = session.get(url, headers=HEADERS, timeout=15)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        if resp.status_code == 200:
            return resp.text
        if resp.status_code == 404:
            return None
        time.sleep(2 * (attempt + 1))
    return None


def strip_html(node):
    if node is None:
        return ""
    return unescape(re.sub(r"\s+", " ", node.get_text(" ", strip=True)))


def parse_article(html, url):
    soup = BeautifulSoup(html, "lxml")

    ld_tag = soup.find("script", {"type": "application/ld+json"})
    claim_review = {}
    if ld_tag and ld_tag.string:
        try:
            claim_review = json.loads(ld_tag.string)
        except json.JSONDecodeError:
            pass

    title = strip_html(soup.find("title")).split(" | ")[0]
    if not title:
        return None

    verdict_raw = (claim_review.get("reviewRating") or {}).get("alternateName")
    verdict = VERDICT_MAP.get((verdict_raw or "").strip().lower(), "UNVERIFIABLE")

    return {
        "source_id": url.rstrip("/").rsplit("/", 1)[-1].split("-", 1)[0],
        "url": url,
        "judul": title,
        "verdict_raw": verdict_raw,
        "verdict": verdict,
        "tanggal": (claim_review.get("itemReviewed") or {}).get("datePublished"),
        "narasi": strip_html(soup.select_one(".article-origin div")),
        "deskripsi": strip_html(soup.select_one(".article-explanation div"))[:500],
        "teks": strip_html(soup.select_one(".article-explanation div")),
        "referensi": "; ".join(a["href"] for a in soup.select(".article-references a[href]")),
    }


def discover_urls_from_listing(pages, session):
    seen = set()
    for page in range(1, pages + 1):
        html = fetch(f"{BASE}/articles?page={page}", session)
        if not html:
            break
        for m in re.finditer(rf'href="({re.escape(BASE)}/articles/\d+-[^"]+)"', html):
            if m.group(1) not in seen:
                seen.add(m.group(1))
                yield m.group(1)
        time.sleep(random.uniform(*DELAY))

## 5. Run the collection

In [ ]:
session = requests.Session()

if START_ID is not None:
    urls = [f"{BASE}/articles/{i}" for i in range(START_ID, END_ID + 1)]
else:
    urls = list(discover_urls_from_listing(PAGES_TO_WALK, session))

print(f"{len(urls)} candidate urls")

rows = []
for url in tqdm(urls):
    html = fetch(url, session)
    time.sleep(random.uniform(*DELAY))
    if html is None:
        continue
    record = parse_article(html, url)
    if record:
        rows.append(record)

df = pd.DataFrame(rows)
print(f"collected {len(df)} articles")
df["verdict"].value_counts()

## 6. Save

Writes to `/kaggle/working/` — after running, use **Save Version > Save & Run All (Commit)**,
then **New Dataset** on the output to publish it, same as your usual habit.

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"saved -> {OUT_CSV}")
df.head()